In [1]:
#Stage 2 Facial feature localization + blink/mouth opening recognition + local texture mapping

In [2]:
import sys
!"{sys.executable}" -m pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import cv2

print(cv2.__version__)

4.10.0


In [4]:
import cv2
import numpy as np 
from matplotlib import pyplot as plt
%matplotlib inline 

# Import required modules
import sys
# Uninstall the broken OpenCV 5 beta versions
!{sys.executable} -m pip uninstall -y cv2 opencv-python opencv-contrib-python
# Force install the latest stable OpenCV 4.10
!{sys.executable} -m pip install opencv-python==4.10.0.84 opencv-contrib-python==4.10.0.84

Found existing installation: opencv-python 4.10.0.84
Uninstalling opencv-python-4.10.0.84:
  Successfully uninstalled opencv-python-4.10.0.84
Found existing installation: opencv-contrib-python 4.10.0.84
Uninstalling opencv-contrib-python-4.10.0.84:
  Successfully uninstalled opencv-contrib-python-4.10.0.84


You can safely remove it manually.


Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)
Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl (45.5 MB)

   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 0/2 [opencv-python]
   -------------------- ------------------- 1/2 [opencv-contrib-python]
   -------------------- ------------------- 1/2 [opencv-contrib-python]
   -------------------- ------------------- 1/2 [opencv-contrib-python]
   -------------------- ------------------- 1/2 [opencv-contrib-python]
   -------------------- ------------------- 1/2 [opencv-con

In [5]:
#Select regions in this order:
#1. Whole head.
#2. Screen-left eye.
#3. Screen-right eye.
#4. Mouth.
#Include eye corners, eyelids and a small skin margin. Avoid eyebrows. Include both lips and a small margin. Press Enter to confirm.

In [6]:
from pathlib import Path

import cv2

import numpy as np

# Input video path
VIDEO_PATH = Path("C:/Users/sili0207/Desktop/AMME5710/Major project/Zelenskyy.mp4")

# Test tracking without a texture first
TEXTURE_PATH = None

# After checking tracking, set a transparent PNG path
# TEXTURE_PATH = Path("cartoon_face.png")

# Shi–Tomasi corner settings
FEATURE_PARAMS = dict(
    maxCorners=150,
    qualityLevel=0.01,
    minDistance=7,
    blockSize=7
)

# Pyramidal Lucas–Kanade optical flow settings
LK_PARAMS = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
)

# Maximum forward-backward error in pixels
FB_THRESHOLD = 1.5

# RANSAC reprojection threshold in pixels
RANSAC_THRESHOLD = 3.0

# Minimum point matches for stable estimation
MIN_POINTS = 10

WINDOW_NAME = "Face tracking: Q quit | R reset | Space pause"

In [7]:
def initialize_tracking(frame):
    """Select the face and initialize points and reference corners."""

    selection_window = "Select face ROI, then press Enter"

    # Drag to select the face. Enter confirms, Esc cancels.
    x, y, w, h = cv2.selectROI(
        selection_window,
        frame,
        showCrosshair=True,
        fromCenter=False
    )
    cv2.destroyWindow(selection_window)

    if w == 0 or h == 0:
        raise ValueError("No valid face region selected.")

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Find points only inside the selected region
    face_mask = np.zeros(gray.shape, dtype=np.uint8)
    face_mask[y:y + h, x:x + w] = 255

    points = cv2.goodFeaturesToTrack(
        gray,
        mask=face_mask,
        **FEATURE_PARAMS
    )

    if points is None or len(points) < MIN_POINTS:
        raise ValueError(
            "Too few face points. Select again or use a clearer video."
        )

    # Reference corner order: top-left, top-right, bottom-right, bottom-left
    reference_corners = np.float32([
        [x, y],
        [x + w - 1, y],
        [x + w - 1, y + h - 1],
        [x, y + h - 1]
    ]).reshape(-1, 1, 2)

    return gray, points, reference_corners


def track_points(previous_gray, current_gray, previous_points,
                 reference_points):
    """Track points and retain their initial reference coordinates."""

    # Forward: previous frame to current frame
    current_points, forward_status, _ = cv2.calcOpticalFlowPyrLK(
        previous_gray,
        current_gray,
        previous_points,
        None,
        **LK_PARAMS
    )

    if current_points is None or forward_status is None:
        return None, None

    # Remove failed forward tracks and invalid coordinates
    forward_valid = (
        (forward_status.ravel() == 1)
        & np.isfinite(current_points.reshape(-1, 2)).all(axis=1)
    )

    if not np.any(forward_valid):
        return None, None

    previous_valid = previous_points[forward_valid]
    current_valid = current_points[forward_valid]
    reference_valid = reference_points[forward_valid]

    # Backward: current frame to previous frame
    backward_points, backward_status, _ = cv2.calcOpticalFlowPyrLK(
        current_gray,
        previous_gray,
        current_valid,
        None,
        **LK_PARAMS
    )

    if backward_points is None or backward_status is None:
        return None, None

    # Check whether backward tracks return to their starting points
    fb_error = np.linalg.norm(
        previous_valid.reshape(-1, 2)
        - backward_points.reshape(-1, 2),
        axis=1
    )

    height, width = current_gray.shape
    xy = current_valid.reshape(-1, 2)

    # Current points must stay inside the image
    inside_frame = (
        (xy[:, 0] >= 0)
        & (xy[:, 0] < width)
        & (xy[:, 1] >= 0)
        & (xy[:, 1] < height)
    )

    reliable = (
        (backward_status.ravel() == 1)
        & np.isfinite(fb_error)
        & (fb_error < FB_THRESHOLD)
        & inside_frame
    )

    return current_valid[reliable], reference_valid[reliable]


def load_texture(texture_path):
    """Load a BGRA texture. Return None if no path is set."""

    if texture_path is None:
        return None

    texture = cv2.imread(str(texture_path), cv2.IMREAD_UNCHANGED)

    if texture is None:
        raise FileNotFoundError(f"Cannot read texture: {texture_path}")

    if texture.ndim != 3 or texture.shape[2] != 4:
        raise ValueError("Texture must be a four-channel BGRA PNG with alpha.")

    if min(texture.shape[:2]) < 2:
        raise ValueError("Texture is too small.")

    return texture


def overlay_texture(frame, texture, texture_to_frame):
    """Warp the texture and alpha, then overlay them on the frame."""

    height, width = frame.shape[:2]

    texture_color = texture[:, :, :3].astype(np.float32)
    texture_alpha = texture[:, :, 3].astype(np.float32) / 255.0

    # Premultiply alpha to reduce dark edges during interpolation
    premultiplied_color = texture_color * texture_alpha[:, :, None]

    warped_color = cv2.warpPerspective(
        premultiplied_color,
        texture_to_frame,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    warped_alpha = cv2.warpPerspective(
        texture_alpha,
        texture_to_frame,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    output = (
        warped_color
        + frame.astype(np.float32) * (1.0 - warped_alpha[:, :, None])
    )

    return np.clip(output, 0, 255).astype(np.uint8)

In [8]:
# Run version 1 cells 1 and 2 before this cell

# Template search margin in reference coordinates
# Values are fractions of the region width and height
SEARCH_MARGIN = 0.35

# Normalized SSD threshold. Lower values are stricter
# Initial setting. Check it on video
MATCH_THRESHOLD = 0.15

REPLENISH_THRESHOLD = 80
NEW_POINT_DISTANCE = 10

PART_NAMES = ["screen_left_eye", "screen_right_eye", "mouth"]


def select_part(frame, name):
    """Manually select a named facial region."""

    window = f"Select {name}, then press Enter"

    x, y, w, h = cv2.selectROI(
        window,
        frame,
        showCrosshair=True,
        fromCenter=False
    )

    cv2.destroyWindow(window)

    if w < 8 or h < 8:
        raise ValueError(f"{name} region is too small or selection was canceled.")

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    return {
        "name": name,
        "rect": (int(x), int(y), int(w), int(h)),
        # Keep the initial template fixed to reduce drift
        "template": gray[y:y + h, x:x + w].copy()
    }


def localize_part(aligned_gray, valid_mask, part):
    """
    Run local template matching in the aligned reference image.
    If matching fails, use the global motion prediction and mark it as unmatched.
    """

    x, y, w, h = part["rect"]
    height, width = aligned_gray.shape

    margin_x = max(4, int(w * SEARCH_MARGIN))
    margin_y = max(4, int(h * SEARCH_MARGIN))

    x0 = max(0, x - margin_x)
    y0 = max(0, y - margin_y)
    x1 = min(width, x + w + margin_x)
    y1 = min(height, y + h + margin_y)

    search_image = aligned_gray[y0:y1, x0:x1]

    # Fall back to the initial position using global motion only
    selected_rect = (x, y, w, h)
    score = float("inf")
    matched = False

    if search_image.shape[0] >= h and search_image.shape[1] >= w:
        response = cv2.matchTemplate(
            search_image,
            part["template"],
            cv2.TM_SQDIFF_NORMED
        )

        # Exclude invalid image-border areas after alignment
        search_valid = (
            valid_mask[y0:y1, x0:x1] > 250
        ).astype(np.float32)

        coverage = cv2.matchTemplate(
            search_valid,
            np.ones((h, w), dtype=np.float32),
            cv2.TM_CCORR
        ) / float(w * h)

        response[coverage < 0.999] = np.inf
        response[~np.isfinite(response)] = np.inf

        if np.isfinite(response).any():
            min_value, _, min_location, _ = cv2.minMaxLoc(response)
            score = float(min_value)

            if score <= MATCH_THRESHOLD:
                selected_rect = (
                    x0 + min_location[0],
                    y0 + min_location[1],
                    w,
                    h
                )
                matched = True

    return selected_rect, score, matched


def rect_corners(rect):
    """Convert a rectangle to four corners."""

    x, y, w, h = rect

    return np.float32([
        [x, y],
        [x + w - 1, y],
        [x + w - 1, y + h - 1],
        [x, y + h - 1]
    ]).reshape(-1, 1, 2)


def replenish_points(gray, points, reference, transform, corners):
    """Add ordinary corners for global head tracking, as in version 1."""

    if len(points) >= REPLENISH_THRESHOLD:
        return points, reference, 0

    mask = np.zeros(gray.shape, dtype=np.uint8)

    cv2.fillConvexPoly(
        mask,
        np.rint(corners).astype(np.int32).reshape(-1, 2),
        255
    )

    mask = cv2.erode(
        mask,
        np.ones((9, 9), dtype=np.uint8)
    )

    for point in points.reshape(-1, 2):
        cv2.circle(
            mask,
            tuple(np.rint(point).astype(int)),
            NEW_POINT_DISTANCE,
            0,
            -1
        )

    params = FEATURE_PARAMS.copy()
    params["maxCorners"] = FEATURE_PARAMS["maxCorners"] - len(points)

    if params["maxCorners"] <= 0:
        return points, reference, 0

    new_points = cv2.goodFeaturesToTrack(
        gray,
        mask=mask,
        **params
    )

    if new_points is None:
        return points, reference, 0

    try:
        new_reference = cv2.perspectiveTransform(
            new_points,
            np.linalg.inv(transform)
        )
    except np.linalg.LinAlgError:
        return points, reference, 0

    valid = np.isfinite(
        new_reference.reshape(-1, 2)
    ).all(axis=1)

    new_points = new_points[valid]
    new_reference = new_reference[valid]

    return (
        np.concatenate([points, new_points], axis=0),
        np.concatenate([reference, new_reference], axis=0),
        len(new_points)
    )


# Release any video capture left from the previous run
if "cap" in globals():
    cap.release()

cap = cv2.VideoCapture(str(VIDEO_PATH))

try:
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

    success, first_frame = cap.read()

    if not success:
        raise RuntimeError("Cannot read the first frame.")

    previous_gray, previous_points, reference_corners = (
        initialize_tracking(first_frame)
    )

    reference_points = previous_points.copy()

    parts = [
        select_part(first_frame, name)
        for name in PART_NAMES
    ]

    frame_height, frame_width = first_frame.shape[:2]

    fps = cap.get(cv2.CAP_PROP_FPS)

    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    display_delay = max(1, int(round(1000.0 / fps)))

    print("Initialization complete.")
    print(f"Head tracking points: {len(previous_points)}")
    print("Run cell 4 next.")

except Exception:
    cap.release()
    cv2.destroyAllWindows()
    raise

Initialization complete.
Head tracking points: 150
Run cell 4 next.


In [9]:
# Place between cells 3 and 4
# Rerun this cell whenever cell 3 is rerun

# Collect both eyes each frame to capture short blinks
eye_samples = []

# Keep the original function to avoid nested wrappers on reruns
original_localize_part = getattr(
    localize_part,
    "_original_function",
    localize_part
)

# Frame index for the facial-region loop
collection_state = {"frame_index": -1}


def localize_part_and_collect(aligned_gray, valid_mask, part):
    """Keep localization unchanged and collect eye crops without text."""

    rect, score, matched = original_localize_part(
        aligned_gray,
        valid_mask,
        part
    )

    # The loop processes the screen-left eye first. Increment once per frame
    if part["name"] == "screen_left_eye":
        collection_state["frame_index"] += 1

    if part["name"] in (
        "screen_left_eye",
        "screen_right_eye"
    ):
        x, y, w, h = rect

        crop = aligned_gray[y:y + h, x:x + w]
        crop_valid = valid_mask[y:y + h, x:x + w]

        # Collect complete crops without invalid warp borders
        valid_crop = (
            crop.shape == (h, w)
            and crop_valid.shape == (h, w)
            and crop.size > 0
            and np.all(crop_valid > 250)
        )

        if valid_crop:
            eye_samples.append({
                "frame_index": collection_state["frame_index"],
                "eye": part["name"],
                "image": crop.copy(),
                "matched": bool(matched),
                "score": float(score),
                "source_video": str(VIDEO_PATH.resolve())
            })

    return rect, score, matched


localize_part_and_collect._original_function = original_localize_part
localize_part = localize_part_and_collect

print("Collection enabled. Run cell 4 now.")

Collection enabled. Run cell 4 now.


In [10]:
SAVE_VIDEO = (
    input("Save version 2 results? Enter y to save: ")
    .strip()
    .lower() == "y"
)

OUTPUT_PATH = VIDEO_PATH.with_name(
    VIDEO_PATH.stem + "_v2_regions.mp4"
)

WINDOW_V2 = "V2 regions | Q quit | Space pause"

# Facial-region preview panel on the right
PANEL_WIDTH = 320
ROW_HEIGHT = max(1, frame_height // 3)

writer = None
written_frames = 0
tracking_active = True

cv2.namedWindow(WINDOW_V2, cv2.WINDOW_NORMAL)

try:
    if SAVE_VIDEO and OUTPUT_PATH.exists():
        SAVE_VIDEO = (
            input("Output exists. Enter y to overwrite: ")
            .strip()
            .lower() == "y"
        )

    if SAVE_VIDEO:
        writer = cv2.VideoWriter(
            str(OUTPUT_PATH),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (frame_width + PANEL_WIDTH, frame_height)
        )

        if not writer.isOpened():
            raise RuntimeError("Cannot create output video.")

    # Include the first frame
    frame = first_frame.copy()
    is_first_frame = True

    while True:
        current_gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        output = frame.copy()

        panel = np.zeros(
            (frame_height, PANEL_WIDTH, 3),
            dtype=np.uint8
        )

        transform = None
        added_count = 0

        if is_first_frame:
            transform = np.eye(3, dtype=np.float64)

        elif tracking_active:
            current_points, matched_reference = track_points(
                previous_gray,
                current_gray,
                previous_points,
                reference_points
            )

            if current_points is None or len(current_points) < MIN_POINTS:
                tracking_active = False

            else:
                affine, status = cv2.estimateAffinePartial2D(
                    matched_reference.reshape(-1, 2),
                    current_points.reshape(-1, 2),
                    method=cv2.RANSAC,
                    ransacReprojThreshold=RANSAC_THRESHOLD,
                    maxIters=2000,
                    confidence=0.99,
                    refineIters=10
                )

                if (
                    affine is None
                    or status is None
                    or not np.isfinite(affine).all()
                ):
                    tracking_active = False

                else:
                    inliers = status.ravel().astype(bool)

                    if np.count_nonzero(inliers) < MIN_POINTS:
                        tracking_active = False

                    else:
                        transform = np.vstack([
                            affine,
                            [0.0, 0.0, 1.0]
                        ])

                        previous_points = current_points[inliers].copy()
                        reference_points = matched_reference[inliers].copy()

        if tracking_active and transform is not None:
            current_corners = cv2.perspectiveTransform(
                reference_corners,
                transform
            )

            xy = current_corners.reshape(-1, 2)

            geometry_valid = np.isfinite(xy).all()

            if geometry_valid:
                geometry_valid = (
                    (np.abs(xy[:, 0]) < 3 * frame_width).all()
                    and
                    (np.abs(xy[:, 1]) < 3 * frame_height).all()
                )

            if geometry_valid:
                area = abs(
                    cv2.contourArea(xy.astype(np.float32))
                )

                geometry_valid = (
                    100 < area < frame_width * frame_height
                )

            if not geometry_valid:
                tracking_active = False

        if tracking_active and transform is not None:
            previous_points, reference_points, added_count = (
                replenish_points(
                    current_gray,
                    previous_points,
                    reference_points,
                    transform,
                    current_corners
                )
            )

            # Current frame to initial reference coordinates
            inverse_transform = np.linalg.inv(transform)

            aligned_gray = cv2.warpAffine(
                current_gray,
                inverse_transform[:2],
                (frame_width, frame_height),
                flags=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_CONSTANT,
                borderValue=0
            )

            # Mark areas with valid source pixels after alignment
            valid_mask = cv2.warpAffine(
                np.full(current_gray.shape, 255, dtype=np.uint8),
                inverse_transform[:2],
                (frame_width, frame_height),
                flags=cv2.INTER_NEAREST,
                borderMode=cv2.BORDER_CONSTANT,
                borderValue=0
            )

            cv2.polylines(
                output,
                [np.rint(current_corners).astype(np.int32)],
                True,
                (255, 255, 0),
                2
            )

            for index, part in enumerate(parts):
                rect, score, matched = localize_part(
                    aligned_gray,
                    valid_mask,
                    part
                )

                # Local region: reference coordinates to current frame
                part_polygon = cv2.perspectiveTransform(
                    rect_corners(rect),
                    transform
                )

                color = (
                    (0, 255, 0)
                    if matched
                    else (0, 165, 255)
                )

                cv2.polylines(
                    output,
                    [np.rint(part_polygon).astype(np.int32)],
                    True,
                    color,
                    2
                )

                x, y, w, h = rect
                crop = aligned_gray[y:y + h, x:x + w]

                # One preview row per region
                row_start = index * ROW_HEIGHT
                row_end = (
                    frame_height
                    if index == len(parts) - 1
                    else (index + 1) * ROW_HEIGHT
                )

                state = "MATCH" if matched else "PREDICTED"
                score_text = (
                    f"{score:.3f}"
                    if np.isfinite(score)
                    else "invalid"
                )

                cv2.putText(
                    panel,
                    part["name"],
                    (10, row_start + 23),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 255, 255),
                    1
                )

                cv2.putText(
                    panel,
                    f"{state} SSD={score_text}",
                    (10, row_start + 46),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    1
                )

                # Display the crop with its aspect ratio preserved
                available_height = row_end - row_start - 65
                available_width = PANEL_WIDTH - 20

                if crop.size > 0 and available_height > 0:
                    preview_scale = min(
                        available_width / crop.shape[1],
                        available_height / crop.shape[0]
                    )

                    preview_width = max(
                        1, int(crop.shape[1] * preview_scale)
                    )
                    preview_height = max(
                        1, int(crop.shape[0] * preview_scale)
                    )

                    preview = cv2.resize(
                        crop,
                        (preview_width, preview_height)
                    )

                    preview = cv2.cvtColor(
                        preview,
                        cv2.COLOR_GRAY2BGR
                    )

                    top = row_start + 60

                    panel[
                        top:top + preview_height,
                        10:10 + preview_width
                    ] = preview

            cv2.putText(
                output,
                f"Head points: {len(previous_points)} | Added: {added_count}",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 255, 0),
                2
            )

        if not tracking_active:
            cv2.putText(
                output,
                "Head tracking lost. Rerun cells 3 and 4.",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 0, 255),
                2
            )

        combined = np.hstack([output, panel])

        if writer is not None:
            writer.write(combined)
            written_frames += 1

        cv2.imshow(WINDOW_V2, combined)

        key = cv2.waitKey(display_delay) & 0xFF

        if key == ord(" "):
            while True:
                key = cv2.waitKey(30) & 0xFF

                if key in (ord(" "), ord("q"), 27):
                    break

                if cv2.getWindowProperty(
                    WINDOW_V2,
                    cv2.WND_PROP_VISIBLE
                ) < 1:
                    key = ord("q")
                    break

        if key in (ord("q"), 27):
            break

        if cv2.getWindowProperty(
            WINDOW_V2,
            cv2.WND_PROP_VISIBLE
        ) < 1:
            break

        previous_gray = current_gray.copy()
        is_first_frame = False

        success, frame = cap.read()

        if not success:
            print("Video processing complete.")
            break

finally:
    cap.release()

    if writer is not None:
        writer.release()
        print(f"Wrote {written_frames} frames: {OUTPUT_PATH}")

    cv2.destroyAllWindows()
print("Eye crops collected: ", len(eye_samples))

Save version 2 results? Enter y to save:  n


Video processing complete.
Eye crops collected:  918


In [11]:
import csv
import uuid
from collections import Counter

if "eye_samples" not in globals() or len(eye_samples) == 0:
    raise RuntimeError(
        "No samples collected. Run cells 3, 3.5 and 4 in order."
    )

# Save beside the input video
DATASET_ROOT = VIDEO_PATH.parent / "eye_dataset"

# Create a new folder each run to avoid overwriting samples
session_id = uuid.uuid4().hex[:12]
SESSION_DIR = DATASET_ROOT / f"{VIDEO_PATH.stem}_{session_id}"

for label in ("open", "closed"):
    (SESSION_DIR / label).mkdir(
        parents=True,
        exist_ok=True
    )

CSV_PATH = SESSION_DIR / "labels.csv"

# Copy the sample list before labeling to keep this session fixed
samples_to_label = list(eye_samples)

# Find the other eye from the same frame for reference only
sample_lookup = {
    (sample["frame_index"], sample["eye"]): sample
    for sample in samples_to_label
}

counts = Counter()
skipped_count = 0

LABEL_WINDOW = "Label LEFT image | 1 open | 0 closed | S skip | Q quit"


def make_eye_preview(image, width=420, height=220):
    """Enlarge the image without stretching and center it on the canvas."""

    canvas = np.full(
        (height, width, 3),
        40,
        dtype=np.uint8
    )

    scale = min(
        width / image.shape[1],
        height / image.shape[0]
    )

    new_width = max(1, int(image.shape[1] * scale))
    new_height = max(1, int(image.shape[0] * scale))

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_NEAREST
    )

    resized = cv2.cvtColor(
        resized,
        cv2.COLOR_GRAY2BGR
    )

    left = (width - new_width) // 2
    top = (height - new_height) // 2

    canvas[
        top:top + new_height,
        left:left + new_width
    ] = resized

    return canvas


def save_png(path, image):
    """Encode before writing to support Unicode Windows paths."""

    success, encoded = cv2.imencode(".png", image)

    if not success:
        raise RuntimeError(f"PNG encoding failed: {path}")

    encoded.tofile(str(path))


cv2.namedWindow(LABEL_WINDOW, cv2.WINDOW_NORMAL)

try:
    with open(
        CSV_PATH,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as csv_file:

        fieldnames = [
            "filename",
            "label",
            "source_video",
            "frame_index",
            "eye",
            "template_matched",
            "template_ssd",
            "session_id"
        ]

        csv_writer = csv.DictWriter(
            csv_file,
            fieldnames=fieldnames
        )

        csv_writer.writeheader()

        for sample_index, sample in enumerate(samples_to_label):
            target_preview = make_eye_preview(sample["image"])

            other_eye = (
                "screen_right_eye"
                if sample["eye"] == "screen_left_eye"
                else "screen_left_eye"
            )

            other_sample = sample_lookup.get(
                (sample["frame_index"], other_eye)
            )

            if other_sample is not None:
                other_preview = make_eye_preview(
                    other_sample["image"]
                )
            else:
                other_preview = np.zeros_like(target_preview)

            view = np.zeros((350, 840, 3), dtype=np.uint8)

            view[70:290, 0:420] = target_preview
            view[70:290, 420:840] = other_preview

            cv2.putText(
                view,
                f"LABEL THIS: {sample['eye']}",
                (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 255),
                1
            )

            cv2.putText(
                view,
                "Other eye: reference only",
                (430, 28),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1
            )

            tracking_state = (
                "MATCH" if sample["matched"] else "PREDICTED"
            )

            cv2.putText(
                view,
                (
                    f"Sample {sample_index + 1}/{len(samples_to_label)}"
                    f" | Frame {sample['frame_index']}"
                    f" | {tracking_state}"
                ),
                (10, 55),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                1
            )

            cv2.putText(
                view,
                (
                    f"1 OPEN | 0 CLOSED | S SKIP | Q QUIT"
                    f"   Saved: open={counts['open']}"
                    f", closed={counts['closed']}"
                ),
                (10, 325),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1
            )

            cv2.imshow(LABEL_WINDOW, view)

            # Wait for a valid label key for this image
            while True:
                key = cv2.waitKey(30) & 0xFF

                if key in (
                    ord("1"), ord("0"),
                    ord("s"), ord(" "),
                    ord("q"), 27
                ):
                    break

                if cv2.getWindowProperty(
                    LABEL_WINDOW,
                    cv2.WND_PROP_VISIBLE
                ) < 1:
                    key = ord("q")
                    break

            if key in (ord("q"), 27):
                break

            if key in (ord("s"), ord(" ")):
                skipped_count += 1
                continue

            label = "open" if key == ord("1") else "closed"

            filename = (
                f"frame_{sample['frame_index']:06d}_"
                f"{sample['eye']}.png"
            )

            relative_path = Path(label) / filename
            output_path = SESSION_DIR / relative_path

            # Save the original crop rather than the enlarged preview
            save_png(output_path, sample["image"])

            csv_writer.writerow({
                "filename": relative_path.as_posix(),
                "label": label,
                "source_video": sample["source_video"],
                "frame_index": sample["frame_index"],
                "eye": sample["eye"],
                "template_matched": sample["matched"],
                "template_ssd": sample["score"],
                "session_id": session_id
            })

            # Write each label immediately to preserve progress on early exit
            csv_file.flush()
            counts[label] += 1

finally:
    cv2.destroyAllWindows()

print("Labeling session complete.")
print("Open-eye samples: ", counts["open"])
print("Closed-eye samples: ", counts["closed"])
print("Skipped samples: ", skipped_count)
print("Output folder: ", SESSION_DIR)

Labeling session complete.
Open-eye samples:  0
Closed-eye samples:  0
Skipped samples:  0
Output folder:  C:\Users\sili0207\Desktop\AMME5710\Major project\eye_dataset\Zelenskyy_6ab4ce409ba2
